# Clustering — Variation 1: Daily Feature Clustering

Groups patient-days into interpretable clusters using engineered daily features: sensor-only features (steps, sleep, HR).  

## Setup & Data Loading

In [8]:
import sys, os, warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.decomposition import PCA
warnings.filterwarnings('ignore')
pd.set_option("display.max_columns", None)

# ── Path setup ────────────────────────────────────────────────────────────────
# Run from project root or notebooks/; adjust paths accordingly
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..') if os.getcwd().endswith('notebooks') else os.getcwd())
sys.path.insert(0, PROJECT_ROOT)

import data_prep as dp
import clustering as cl

DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
EXT_DIR  = os.path.join(DATA_DIR, 'external')

DISEASE_ORDER   = ['Early Disease Stage', 'Fast Disease Progression', 'Late Disease Stage']
DISEASE_PALETTE = {'Early Disease Stage': '#2ecc71', 'Fast Disease Progression': '#e74c3c', 'Late Disease Stage': '#3498db'}


In [9]:
import glob

files = glob.glob(os.path.join(DATA_DIR, 'Hourly Sensor Data', 'RHourly_*.csv'))
chunks = []
for fp in files:
    pid = int(os.path.basename(fp).replace('RHourly_', '').replace('.csv', ''))
    c = pd.read_csv(fp); c['id'] = pid; chunks.append(c)
sensor = pd.concat(chunks, ignore_index=True)
sensor['time']      = pd.to_datetime(sensor['time'])
sensor['heartrate'] = pd.to_numeric(sensor['heartrate'], errors='coerce')

clinical = pd.read_csv(os.path.join(DATA_DIR, 'ClinicalMarkers_final.csv'))
clinical.columns = clinical.columns.str.strip().str.lower().str.replace('.', '_', regex=False)
clinical['disease_type'] = clinical['disease_type'].str.strip()
df_merged = clinical[['id', 'age', 'sex', 'disease_type']].merge(sensor, on='id', how='left')

df = dp.get_complete_patient_days(df_merged, shift_hour=6, shifted_columns=['sleep'], complete=True)
print(f'Complete rows: {len(df):,}  |  patients: {df["id"].nunique()}')


Complete rows: 46,259  |  patients: 43


In [10]:
df

,id,age,sex,disease_type,time,steps,sleep,heartrate,shifted_sleep,n_complete_days
35207,1120,1982,Female,Late Disease Stage,2021-06-09 00:00:00,0.00000,0,72.097265,0.0,52
35208,1120,1982,Female,Late Disease Stage,2021-06-09 01:00:00,0.00000,0,76.176299,0.0,52
35209,1120,1982,Female,Late Disease Stage,2021-06-09 02:00:00,0.00000,0,78.143429,0.0,52
35210,1120,1982,Female,Late Disease Stage,2021-06-09 03:00:00,0.00000,45,76.017211,0.0,52
35211,1120,1982,Female,Late Disease Stage,2021-06-09 04:00:00,0.00000,60,73.999163,0.0,52
...,...,...,...,...,...,...,...,...,...,...
24677,9926,1985,Female,Late Disease Stage,2021-06-07 17:00:00,4055.95700,0,107.751611,0.0,39
24678,9926,1985,Female,Late Disease Stage,2021-06-07 18:00:00,253.45140,0,92.062437,0.0,39
24679,9926,1985,Female,Late Disease Stage,2021-06-07 19:00:00,251.89790,0,80.565961,0.0,39
24680,9926,1985,Female,Late Disease Stage,2021-06-07 20:00:00,131.90584,0,85.548315,0.0,39


## Create features
- aggregate into daily level
- HR related features: z-score (taking into account individual baseline)
- The rest: standard scaler across all patients

In [11]:
daily = dp.aggregate_to_daily(df)
print(f'Daily shape: {daily.shape}')


Computing per-day features...
Daily shape: (1933, 29)


In [12]:
weather = dp.load_weather(os.path.join(EXT_DIR, 'weather', 'ogd-smn_klo_h_historical_2020-2029.csv'), daily=True)
cal     = dp.load_calendar(os.path.join(EXT_DIR, 'holidays', 'zurich_calendar_2021.csv'))
covid   = dp.load_covid(os.path.join(EXT_DIR, 'covid', 'ch_stringency_2021.csv'))
pollen  = dp.load_pollen(os.path.join(EXT_DIR, 'pollen', 'ogd-pollen_pzh_d_historical.csv'))

daily = daily.merge(weather, on='date', how='left')
daily = daily.merge(cal[['date', 'is_weekend', 'day_type']], on='date', how='left')
daily = daily.merge(covid[['date', 'stringency_index']], on='date', how='left')
daily = daily.merge(pollen[['date', 'pollen_total']], on='date', how='left')
# is_weekend as int for scaling
daily['is_weekend'] = daily['is_weekend'].astype(float)
print(f'Daily with external: {daily.shape}')

Daily with external: (1933, 39)


In [13]:
daily.describe()

,id,date,n_complete_days,age,daily_steps,active_hours,sedentary_hours,step_peak,peak_steps_hour,step_entropy,sum_sleep_minute,sleep_fragmentation,sleep_fragmentation_min,sleep_onset_hour,sleep_end_hour,mean_hr,max_hr,min_hr,resting_hr,day_hr,day_hr_var,resting_hr_var,hr_day_night_delta,ncc,ncc_per_step,no_active_hour,restless_night,temp_max,temp_mean,temp_min,precip_total,sunshine_total,humidity_mean,is_weekend,stringency_index,pollen_total
count,1933.000000,1933,1933.000000,1933.000000,1933.000000,1933.000000,1933.000000,1933.000000,1933.000000,1933.000000,1933.00000,1933.000000,1933.000000,1722.000000,1722.000000,1933.000000,1933.000000,1933.000000,1915.000000,1932.000000,1931.000000,1913.000000,1914.000000,1557.000000,1557.000000,1933.000000,1933.000000,1933.000000,1933.000000,1933.000000,1933.000000,1933.000000,1933.000000,1933.000000,1933.000000,1650.000000
mean,5635.821521,2021-06-19 16:27:48.701500416,47.966374,1977.417486,7606.620710,17.900155,6.031040,1740.008575,12.758407,2.400666,480.70150,2.239007,101.047077,4.196864,7.196283,78.603261,98.704203,62.854740,80.516110,84.691376,7.507424,10.304941,4.085885,10.493246,0.009194,0.187274,0.009312,17.312985,11.714270,6.346094,3.001707,336.669943,75.960618,0.281428,51.423378,63.992727
min,1120.000000,2021-01-09 00:00:00,1.000000,1958.000000,137.064649,3.000000,0.000000,80.016792,0.000000,0.943807,0.00000,0.000000,0.000000,0.000000,0.000000,62.140447,72.968372,45.463492,59.398864,64.127513,1.295683,0.491925,-12.767754,-9.193041,-0.005995,0.000000,0.000000,-5.200000,-6.866667,-14.800000,0.000000,0.000000,43.537500,0.000000,44.440000,0.000000
25%,3389.000000,2021-04-13 00:00:00,43.000000,1972.000000,3755.895789,16.000000,5.000000,731.284790,9.000000,2.268605,431.00000,2.000000,42.000000,3.000000,6.000000,73.071463,90.961705,57.664908,73.652309,78.735493,5.623297,7.604872,1.486940,6.693155,0.005807,0.000000,0.000000,12.500000,6.833333,1.100000,0.000000,95.000000,67.650000,0.000000,48.150000,4.000000
50%,5977.000000,2021-06-13 00:00:00,49.000000,1978.000000,6758.110474,18.000000,6.000000,1397.403900,13.000000,2.445300,490.00000,2.000000,63.000000,4.000000,6.000000,77.496949,97.691338,61.567171,79.331672,83.436872,7.127265,10.168114,4.103300,10.146338,0.008331,0.000000,0.000000,17.800000,11.550000,6.900000,0.000000,286.000000,77.354167,0.000000,50.000000,19.000000
75%,7513.000000,2021-09-05 00:00:00,52.000000,1982.000000,10499.655703,19.000000,7.000000,2318.265100,15.000000,2.572395,546.00000,2.000000,90.000000,5.000000,6.000000,82.177188,105.732006,66.906223,85.230911,89.198942,8.988834,12.804736,6.561728,13.581351,0.011724,0.000000,0.000000,22.600000,16.983333,11.900000,3.100000,543.000000,85.075000,1.000000,60.190000,85.000000
max,9926.000000,2021-11-14 00:00:00,74.000000,2005.000000,29545.063499,24.000000,18.000000,6088.826200,23.000000,2.951835,1042.00000,7.000000,775.000000,23.000000,23.000000,110.182995,139.439355,97.115911,117.439153,117.672698,23.043989,25.235674,29.053622,45.063626,0.042599,1.000000,1.000000,31.300000,24.654167,18.400000,60.300000,892.000000,99.275000,1.000000,60.190000,848.000000
std,2494.382741,NaN,8.611671,8.677214,4826.959282,2.280729,2.173401,1308.179595,3.880872,0.245314,116.53921,0.752353,127.708893,2.102855,4.189557,8.208007,10.902226,7.624109,9.589981,9.148663,2.663912,3.786085,4.258047,5.741549,0.005553,0.390232,0.096073,7.061824,6.296255,6.345342,6.857476,266.646517,11.250740,0.449812,5.761186,108.575321


In [14]:
daily = daily[daily['sum_sleep_minute']>0]  # Exclude days with no sleep data (likely device not worn)
feat_df = dp.build_daily_features(daily, include_external=False)
feature_cols = [c for c in feat_df.columns if c.endswith(('_sc', '_z', '_bool'))]
keep_cols    = ['id', 'date'] + [c for c in ['disease_type', 'sex', 'age'] if c in feat_df.columns]
print(f'Before imputation: {feat_df[feature_cols].isna().sum().sum()} NaN values')
feat_df[feature_cols].isna().sum()[lambda s: s > 0]

Before imputation: 734 NaN values


resting_hr_z              2
day_hr_z                  2
day_hr_var_z              3
resting_hr_var_z          4
hr_day_night_delta_z      3
ncc_z                   360
ncc_per_step_z          360
dtype: int64

## Impute missing values

In [15]:
# ── Impute NaN z-scores ───────────────────────────────────────────────────────
feat_df = feat_df.sort_values(['id', 'date']).copy()

# ncc_z, ncc_per_step_z → 2: sedentary days have no active HR signal;
# setting to +2 std places them distinctly above the average active day
for col in ['ncc_z', 'ncc_per_step_z']:
    if col in feat_df.columns:
        feat_df[col] = feat_df[col].fillna(2)

# resting/delta HR → carry last known value forward per patient;
# sleep HR baseline is stable day-to-day
for col in ['resting_hr_z', 'resting_hr_var_z', 'hr_day_night_delta_z']:
    if col in feat_df.columns:
        feat_df[col] = feat_df.groupby('id')[col].ffill()
        feat_df[col] = feat_df.groupby('id')[col].bfill()

# day_hr_z, day_hr_var_z → 0: missing daytime HR treated as typical
for col in ['day_hr_z', 'day_hr_var_z']:
    if col in feat_df.columns:
        feat_df[col] = feat_df[col].fillna(0)

print(f'After imputation:  {feat_df[feature_cols].isna().sum().sum()} NaN values remaining')

# ── Extract meta + feature matrix ─────────────────────────────────────────────
feat_df_clean = feat_df.dropna(subset=feature_cols).reset_index(drop=True)
meta = feat_df_clean[keep_cols]
X    = feat_df_clean[feature_cols].to_numpy(dtype=float)
print(f'Feature matrix: {X.shape}  ({len(feat_df) - len(feat_df_clean)} rows dropped)')
print(f'Features: {feature_cols}')

After imputation:  3 NaN values remaining
Feature matrix: (1915, 15)  (1 rows dropped)
Features: ['resting_hr_z', 'day_hr_z', 'day_hr_var_z', 'resting_hr_var_z', 'hr_day_night_delta_z', 'ncc_z', 'ncc_per_step_z', 'no_active_hour_bool', 'restless_night_bool', 'daily_steps_sc', 'active_hours_sc', 'step_peak_sc', 'step_entropy_sc', 'sum_sleep_minute_sc', 'sleep_fragmentation_min_sc']


In [19]:
feat_df[keep_cols+feature_cols].describe()

,id,date,age,resting_hr_z,day_hr_z,day_hr_var_z,resting_hr_var_z,hr_day_night_delta_z,ncc_z,ncc_per_step_z,no_active_hour_bool,restless_night_bool,daily_steps_sc,active_hours_sc,step_peak_sc,step_entropy_sc,sum_sleep_minute_sc,sleep_fragmentation_min_sc
count,1916.000000,1916,1916.000000,1915.000000,1.916000e+03,1.916000e+03,1915.000000,1915.000000,1916.000000,1916.000000,1916.000000,1916.000000,1.916000e+03,1.916000e+03,1.916000e+03,1.916000e+03,1.916000e+03,1.916000e+03
mean,5658.449374,2021-06-19 15:34:56.868475904,1977.377871,-0.001395,3.418745e-16,2.966776e-17,0.000778,0.001327,0.375783,0.375783,0.186848,0.000522,-2.373420e-16,-6.675245e-16,-2.225082e-17,-5.191857e-16,-1.483388e-16,-4.079316e-17
min,1120.000000,2021-01-09 00:00:00,1958.000000,-2.811591,-3.331118e+00,-2.846389e+00,-3.195783,-4.411316,-3.282995,-3.306536,0.000000,0.000000,-1.554432e+00,-6.547167e+00,-1.271617e+00,-5.933627e+00,-4.293659e+00,-7.971560e-01
25%,3389.000000,2021-04-13 00:00:00,1972.000000,-0.717216,-6.858103e-01,-7.084762e-01,-0.705319,-0.641438,-0.524866,-0.528638,0.000000,0.000000,-8.022480e-01,-8.325917e-01,-7.728645e-01,-5.390081e-01,-4.749945e-01,-4.609142e-01
50%,5977.000000,2021-06-12 12:00:00,1978.000000,-0.051244,-5.293696e-02,-7.817095e-02,-0.028009,0.017118,0.216297,0.143607,0.000000,0.000000,-1.773126e-01,4.657374e-02,-2.623205e-01,1.743949e-01,5.595517e-02,-3.045227e-01
75%,7513.000000,2021-09-06 00:00:00,1982.000000,0.625588,6.634451e-01,6.217120e-01,0.677561,0.635684,1.464905,1.494331,0.000000,0.000000,6.068693e-01,4.861565e-01,4.497835e-01,7.010099e-01,5.753121e-01,-9.339415e-02
max,9926.000000,2021-11-14 00:00:00,2005.000000,4.074693,4.040521e+00,4.277359e+00,3.517600,4.340009,3.848313,4.827060,1.000000,1.000000,4.566236e+00,2.684070e+00,3.328127e+00,2.247710e+00,5.166056e+00,5.263016e+00
std,2486.070302,NaN,8.676381,0.990849,9.887091e-01,9.884450e-01,0.988643,0.990602,1.183753,1.183753,0.389891,0.022846,1.000261e+00,1.000261e+00,1.000261e+00,1.000261e+00,1.000261e+00,1.000261e+00


## Fit model
- K-mean

In [75]:
SELECTED_FEATS = [
    'resting_hr_z',
    # 'day_hr_z',
    # 'day_hr_var_z',
    # 'resting_hr_var_z',
    'hr_day_night_delta_z',
    # 'ncc_z',
    'ncc_per_step_z',
    'no_active_hour_bool',
    # 'restless_night_bool',
    'daily_steps_sc',
    # 'active_hours_sc',
    'step_peak_sc',
    # 'step_entropy_sc',
    # 'sum_sleep_minute_sc',
    'sleep_fragmentation_min_sc'
    ]

In [76]:
# Filter to selected features only
feat_idx = [feature_cols.index(f) for f in SELECTED_FEATS if f in feature_cols]
X_r1    = X[:, feat_idx]
feat_r1 = [feature_cols[i] for i in feat_idx]

print(f'Full feature matrix: {X.shape}')
print(f'Selected feature matrix: {X_r1.shape}')
print(f'Selected features: {feat_r1}')
print(f'Rows dropped (NaN): {len(daily) - len(meta)}')


Full feature matrix: (1915, 15)
Selected feature matrix: (1915, 7)
Selected features: ['resting_hr_z', 'hr_day_night_delta_z', 'ncc_per_step_z', 'no_active_hour_bool', 'daily_steps_sc', 'step_peak_sc', 'sleep_fragmentation_min_sc']
Rows dropped (NaN): 1


In [77]:
print('Evaluating k=2..10 (Round 1)...')
eval_r1 = cl.evaluate_k(X_r1, range(2, 11), n_init=10)

fig = make_subplots(rows=1, cols=2,
    subplot_titles=['Elbow — Inertia', 'Silhouette Score (higher = better)'])
fig.add_trace(go.Scatter(x=eval_r1['k'], y=eval_r1['inertia'],
    mode='lines+markers', name='Inertia', line=dict(color='#3498db', width=2)), row=1, col=1)
fig.add_trace(go.Scatter(x=eval_r1['k'], y=eval_r1['silhouette'],
    mode='lines+markers', name='Silhouette', line=dict(color='#e74c3c', width=2)), row=1, col=2)
fig.update_xaxes(title_text='k (clusters)', dtick=1)
fig.update_layout(title='Round 1 (sensor-only): Cluster Count Evaluation',
    height=380, width=900, showlegend=False)
fig.show()
eval_r1


Evaluating k=2..10 (Round 1)...
  k=2: inertia=8907.9, silhouette=0.2644
  k=3: inertia=7422.1, silhouette=0.2417
  k=4: inertia=6149.8, silhouette=0.2661
  k=5: inertia=5247.4, silhouette=0.2547
  k=6: inertia=4830.3, silhouette=0.2354
  k=7: inertia=4505.5, silhouette=0.2160
  k=8: inertia=4227.4, silhouette=0.2212
  k=9: inertia=3968.8, silhouette=0.2115
  k=10: inertia=3781.4, silhouette=0.2072


,k,inertia,silhouette
0,2,8907.872897,0.264429
1,3,7422.147298,0.241683
2,4,6149.837756,0.266105
3,5,5247.429720,0.254697
4,6,4830.301136,0.235391
5,7,4505.457338,0.215955
6,8,4227.396816,0.221156
7,9,3968.841806,0.211512
8,10,3781.417453,0.207204


In [78]:
# ── Set k after inspecting the elbow / silhouette curves above ────────────────
K_R1 = 4  # adjust based on the plots

labels_r1, model_r1 = cl.fit_kmeans(X_r1, k=K_R1)
meta['cluster'] = labels_r1
print(f'Cluster sizes:\n{pd.Series(labels_r1).value_counts().sort_index().to_string()}')


KMeans k=4: inertia=6149.7, silhouette=0.2664
Cluster sizes:
0    700
1    478
2    147
3    590


## Visualization

In [79]:
pca = PCA(n_components=2, random_state=42)
Z   = pca.fit_transform(X_r1)
pca_df = meta.copy()
pca_df['PC1'] = Z[:, 0]
pca_df['PC2'] = Z[:, 1]
pca_df['cluster_str'] = 'Cluster ' + pca_df['cluster'].astype(str)

fig = px.scatter(
    pca_df, x='PC1', y='PC2',
    color='cluster_str',
    symbol='disease_type' if 'disease_type' in pca_df.columns else None,
    hover_data={'id': True, 'date': '|%Y-%m-%d', 'cluster': True,
                'disease_type': True} if 'disease_type' in pca_df.columns else {'id': True},
    title=f'Round 1 — PCA (2D): {K_R1} clusters, sensor-only features',
    labels={'cluster_str': 'Cluster', 'PC1': f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)',
            'PC2': f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)'},
    width=820, height=520,
)
fig.show()


In [80]:
# Radar chart of cluster centroids (raw daily values for interpretability)
desc_r1 = cl.describe_clusters(meta, labels_r1, daily, feat_r1)

# Derive radar features from selected features (strip _z/_sc suffixes)
radar_features = [cl._raw_name(f) for f in feat_r1 if f'mean_{cl._raw_name(f)}' in desc_r1.columns]

# Min-max normalise centroid values for radar display
radar_vals = desc_r1[[f'mean_{f}' for f in radar_features]].copy()
radar_norm = (radar_vals - radar_vals.min()) / (radar_vals.max() - radar_vals.min()).replace(0, 1)

CLUSTER_COLORS = px.colors.qualitative.Plotly

fig = go.Figure()
for i, cl_id in enumerate(radar_norm.index):
    vals = radar_norm.loc[cl_id].tolist()
    vals += [vals[0]]  # close polygon
    fig.add_trace(go.Scatterpolar(
        r=vals, theta=radar_features + [radar_features[0]],
        fill='toself', opacity=0.6,
        name=f'Cluster {cl_id} (n={int(desc_r1.loc[cl_id, "n_days"])})',
        line=dict(color=CLUSTER_COLORS[i % len(CLUSTER_COLORS)]),
    ))
fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
    title=f'Round 1 — Cluster Centroid Profiles (min-max normalised)',
    height=520, width=620,
)
fig.show()


In [81]:
# Summary table — raw feature means per cluster
# (desc_r1 stores raw column names; feature_cols has _sc/_z suffixes — don't mix them)
mean_cols    = [col for col in desc_r1.columns if col.startswith('mean_')]
disease_cols = [col for col in ['pct_early', 'pct_fast', 'pct_late'] if col in desc_r1.columns]
display_cols = ['n_days', 'pct_total'] + mean_cols + disease_cols
desc_r1[display_cols].round(2)


,n_days,pct_total,mean_resting_hr,mean_hr_day_night_delta,mean_ncc_per_step,mean_daily_steps,mean_step_peak,mean_sleep_fragmentation_min,pct_early,pct_fast,pct_late
cluster,,,,,,,,,,,
0,700,36.55,76.99,4.86,0.02,4300.71,862.28,69.35,27.43,14.43,58.14
1,478,24.96,82.10,6.17,0.01,13696.01,3490.03,72.65,59.41,20.50,20.08
2,147,7.68,78.55,2.90,0.01,5103.49,1202.66,501.83,17.69,15.65,66.67
3,590,30.81,83.91,1.77,0.01,7208.18,1496.00,64.77,32.88,15.08,52.03


In [82]:
# ---- Feature importance: one-way ANOVA F-statistic across clusters ----
# High F = this feature differs most between clusters (drives separation).
feat_imp = cl.feature_importance_anova(X_r1, labels_r1, feat_r1)

fig = px.bar(
    feat_imp, x='F_stat', y='raw_feature', orientation='h',
    color='F_stat', color_continuous_scale='Reds',
    error_x=None,
    hover_data={'feature': True, 'p_value': ':.4f'},
    labels={'raw_feature': 'Feature', 'F_stat': 'ANOVA F-statistic'},
    title='Round 1 — Feature Importance: Which features drive cluster separation?',
    width=700, height=420,
)
fig.update_layout(yaxis=dict(categoryorder='total ascending'), coloraxis_showscale=False)
fig.show()
feat_imp[['raw_feature', 'feature', 'F_stat', 'p_value']].round(3)


,raw_feature,feature,F_stat,p_value
0,sleep_fragmentation_min,sleep_fragmentation_min_sc,2770.165,0.0
1,step_peak,step_peak_sc,1123.923,0.0
2,daily_steps,daily_steps_sc,943.304,0.0
3,ncc_per_step,ncc_per_step_z,879.994,0.0
4,no_active_hour_bool,no_active_hour_bool,250.389,0.0
5,resting_hr,resting_hr_z,169.770,0.0
6,hr_day_night_delta,hr_day_night_delta_z,155.252,0.0


In [83]:
# Cluster composition: x = disease stage, y = patient-days, colour = cluster
if 'disease_type' in meta.columns:
    comp = meta.groupby(['disease_type', 'cluster']).size().reset_index(name='n')
    comp['cluster_str'] = 'Cluster ' + comp['cluster'].astype(str)

    # Count chart
    fig = px.bar(
        comp, x='disease_type', y='n', color='cluster_str',
        color_discrete_sequence=CLUSTER_COLORS,
        category_orders={'disease_type': DISEASE_ORDER},
        title='Round 1 — Patient-days per Disease Stage (count)',
        labels={'disease_type': 'Disease Stage', 'n': 'Patient-days', 'cluster_str': 'Cluster'},
        barmode='stack', width=720, height=420,
    )
    fig.show()

    # Percentage chart — within each disease stage, what % of days per cluster?
    total_per_stage = comp.groupby('disease_type')['n'].transform('sum')
    comp['pct'] = comp['n'] / total_per_stage * 100

    fig2 = px.bar(
        comp, x='disease_type', y='pct', color='cluster_str',
        color_discrete_sequence=CLUSTER_COLORS,
        category_orders={'disease_type': DISEASE_ORDER},
        title='Round 1 — Cluster Distribution within each Disease Stage (%)',
        labels={'disease_type': 'Disease Stage', 'pct': '% of days in stage', 'cluster_str': 'Cluster'},
        barmode='stack', width=720, height=420,
        text=comp['pct'].round(1).astype(str) + '%',
    )
    fig2.update_traces(textposition='inside', textfont_size=11)
    fig2.update_layout(yaxis_range=[0, 100])
    fig2.show()


In [84]:
# Calendar heatmap: patient x date, discrete colour per cluster
cal_df = meta.copy()
if 'disease_type' not in cal_df.columns:
    cal_df = cal_df.merge(daily[['id','date','disease_type']], on=['id','date'], how='left')

patient_order = (
    cal_df.groupby(['id','disease_type'])['cluster'].count().reset_index()
    .sort_values(['disease_type','id'])['id'].tolist()
) if 'disease_type' in cal_df.columns else sorted(cal_df['id'].unique())

pivot = cal_df.pivot(index='id', columns='date', values='cluster').reindex(patient_order)

# Step-function colorscale: each cluster band maps to the same colour as disease-bar / relative-day
colorscale = []
for i in range(K_R1):
    lo = i / K_R1
    hi = (i + 1) / K_R1
    colorscale.append([lo, CLUSTER_COLORS[i % len(CLUSTER_COLORS)]])
    colorscale.append([hi, CLUSTER_COLORS[i % len(CLUSTER_COLORS)]])

fig = go.Figure(go.Heatmap(
    z=pivot.values,
    x=[str(d)[:10] for d in pivot.columns],
    y=[str(pid) for pid in pivot.index],
    colorscale=colorscale,
    zmin=-0.5, zmax=K_R1 - 0.5,
    colorbar=dict(
        title='Cluster',
        tickvals=list(range(K_R1)),
        ticktext=[f'Cluster {k}' for k in range(K_R1)],
        lenmode='fraction', len=0.6,
    ),
    hovertemplate='Date: %{x}<br>Patient: %{y}<br>Cluster: %{z}<extra></extra>',
))
fig.update_layout(
    title='Round 1 — Calendar Heatmap: Cluster Assignment per Patient-day',
    height=700, width=1100,
    xaxis=dict(title='Date', tickangle=-45),
    yaxis=dict(title='Patient ID', autorange='reversed'),
)
fig.show()


In [85]:
# Relative-day coverage grid coloured by cluster
# x = study day relative to each patient's first complete day
# y = patient, ordered by disease stage then total days (descending)

_stage_abbr = {
    'Early Disease Stage':      'Early',
    'Fast Disease Progression': 'Fast',
    'Late Disease Stage':       'Late',
}

_first = meta.groupby('id')['date'].min().rename('first_date').reset_index()
meta_rd = meta.merge(_first, on='id')
meta_rd['relative_day'] = (meta_rd['date'] - meta_rd['first_date']).dt.days + 1

patient_info = (
    meta_rd.groupby('id')
    .agg(disease_type=('disease_type', 'first'), n_days=('date', 'nunique'))
    .reset_index()
)
_stage_rank = {s: i for i, s in enumerate(DISEASE_ORDER)}
_id_order = (
    patient_info
    .assign(_r=patient_info['disease_type'].map(_stage_rank))
    .sort_values(['_r', 'n_days'], ascending=[True, False])['id']
    .tolist()
)
_y_pos    = {pid: i for i, pid in enumerate(_id_order)}
_y_labels = [
    f'{pid} ({_stage_abbr.get(patient_info.loc[patient_info["id"]==pid, "disease_type"].values[0], "?")})'
    for pid in _id_order
]

meta_rd['y_pos'] = meta_rd['id'].map(_y_pos)

fig = go.Figure()
for k in range(K_R1):
    sub = meta_rd[meta_rd['cluster'] == k]
    fig.add_trace(go.Scatter(
        x=sub['relative_day'],
        y=sub['y_pos'],
        mode='markers',
        marker=dict(symbol='square', size=8, color=CLUSTER_COLORS[k % len(CLUSTER_COLORS)]),
        name=f'Cluster {k}',
        customdata=sub[['id', 'disease_type']].values,
        hovertemplate=(
            'Patient %{customdata[0]}<br>'
            'Relative day: %{x}<br>'
            f'Cluster: {k}<br>'
            'Stage: %{customdata[1]}'
            '<extra></extra>'
        ),
    ))

# Horizontal dividers between disease stage groups
_stage_sizes = [
    sum(1 for pid in _id_order
        if patient_info.loc[patient_info['id'] == pid, 'disease_type'].values[0] == s)
    for s in DISEASE_ORDER
]
_cum = 0
_shapes, _annotations = [], []
for stage, sz in zip(DISEASE_ORDER, _stage_sizes):
    _annotations.append(dict(
        x=1.01, xref='paper', y=_cum + sz / 2,
        text=_stage_abbr[stage], showarrow=False,
        font=dict(size=10), xanchor='left',
    ))
    _cum += sz
    if _cum < len(_id_order):  # no divider after last group
        _shapes.append(dict(
            type='line', x0=0, x1=1, xref='paper',
            y0=_cum - 0.5, y1=_cum - 0.5,
            line=dict(color='black', width=1, dash='dot'),
        ))

fig.update_layout(
    title='Round 1 — Relative Study-Day Coverage (coloured by cluster)',
    height=700, width=1000,
    shapes=_shapes,
    annotations=_annotations,
    xaxis=dict(title='Relative study day', dtick=7),
    yaxis=dict(
        tickvals=list(range(len(_id_order))),
        ticktext=_y_labels,
        autorange='reversed',
        tickfont=dict(size=9),
    ),
)
fig.show()


In [86]:
from sklearn.metrics import pairwise_distances_argmin

centroid_idx = pairwise_distances_argmin(model_r1.cluster_centers_, X_r1)

# Pre-fetch representative days
rep_days = []
for k in range(K_R1):
    row_meta = meta.iloc[centroid_idx[k]]
    pid, day = row_meta['id'], row_meta['date']
    day_df = (
        df[(df['id'] == pid) & (df['time'].dt.normalize() == pd.Timestamp(day))]
        .sort_values('time')
        .copy()
    )
    rep_days.append((k, pid, day, day_df))

# Universal axis ranges across all clusters
max_steps = max(d['steps'].max() for _, _, _, d in rep_days)
hr_all    = pd.concat([d['heartrate'].dropna() for _, _, _, d in rep_days])
hr_range  = [hr_all.min() * 0.95, hr_all.max() * 1.05]

top_titles = [f'Cluster {k}' for k, _, _, _ in rep_days]
bot_titles = [f'P{pid}  {str(day)[:10]}' for _, pid, day, _ in rep_days]

specs = [[{'secondary_y': True}] * K_R1, [{'secondary_y': False}] * K_R1]
fig = make_subplots(
    rows=2, cols=K_R1,
    subplot_titles=top_titles + bot_titles,
    specs=specs,
    vertical_spacing=0.18, horizontal_spacing=0.08,
)

for k, pid, day, day_df in rep_days:
    clock_labels       = [f"{t.hour:02d}:00" for t in day_df['time']]
    clock_labels_sleep = [f"{(t.hour - 6) % 24:02d}:00" for t in day_df['time']]
    col = k + 1
    show_legend = (k == 0)

    fig.add_trace(go.Bar(
        x=clock_labels, y=day_df['steps'],
        name='Steps/hr', marker_color='#3498db',
        showlegend=show_legend,
    ), row=1, col=col, secondary_y=False)

    fig.add_trace(go.Scatter(
        x=clock_labels, y=day_df['heartrate'],
        mode='lines+markers', marker=dict(size=3),
        name='HR (bpm)', line=dict(color='#e74c3c', width=1.5),
        showlegend=show_legend,
    ), row=1, col=col, secondary_y=True)

    fig.add_trace(go.Bar(
        x=clock_labels_sleep, y=day_df['shifted_sleep'],
        name='Sleep min/hr', marker_color='#9b59b6',
        showlegend=show_legend,
    ), row=2, col=col)

    # Universal ranges
    fig.update_yaxes(range=[0, max_steps * 1.05], row=1, col=col, secondary_y=False)
    fig.update_yaxes(range=hr_range, row=1, col=col, secondary_y=True)

    for r in (1, 2):
        fig.update_xaxes(tickangle=-60, tickfont_size=8, row=r, col=col)

fig.update_yaxes(title_text='Steps / hr', row=1, col=1, secondary_y=False)
fig.update_yaxes(title_text='HR (bpm)', row=1, col=K_R1, secondary_y=True)
fig.update_yaxes(title_text='Sleep min / hr', row=2, col=1)

fig.update_layout(
    title='Round 1 — Representative Day per Cluster (closest to centroid)',
    height=520, width=max(900, 280 * K_R1),
    bargap=0.05,
)
fig.show()


## Export Cluster Labels

In [87]:
label_df_r1 = cl.labels_to_dataframe(meta, labels_r1, label_col='cluster_v1')
label_df_r1.to_csv(os.path.join(DATA_DIR, 'cluster_labels_v1.csv'), index=False)
print(f'Saved cluster_labels_v1.csv: {label_df_r1.shape}')
label_df_r1.head()


Saved cluster_labels_v1.csv: (1915, 3)


,id,date,cluster_v1
0,1120,2021-06-09,3
1,1120,2021-06-10,0
2,1120,2021-06-11,3
3,1120,2021-06-13,0
4,1120,2021-06-14,0
